# Skeleton-Based Action Recognition: TCN vs BiLSTM

Clip classification from pose landmarks. Two sequence models are trained on the same features and the same splits and then compared.

**Pipeline:** video -> per-frame landmarks (59 keypoints, x/y/confidence, 15 FPS) -> fixed 172-frame clips (`.npy`) -> features (position + velocity, 236 dims per frame) -> model -> class logits.

| Section | Contents |
|---|---|
| Skeleton Dataset & Loading | Feature engineering, stratified train/val/test split, class weights |
| TCN Model | Dilated residual TCN with temporal self-attention |
| BiLSTM Model | Bidirectional LSTM with self-attention (comparison baseline) |
| Evaluation | Test-set metrics, per-class F1, confusion matrix, model comparison |

In [1]:
import pandas as pd
import numpy as np
import os
import time
import matplotlib.pyplot as plt

from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

### Plot style

One shared style is defined up front so every figure in the notebook — class balance, training curves, the TCN-vs-BiLSTM comparison, and the evaluation plots — looks consistent. `PALETTE` gives named colors for the two models so the same color always means the same model.

In [ ]:
plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.edgecolor":    "#444444",
    "axes.grid":         True,
    "grid.color":        "#dddddd",
    "grid.linewidth":    0.6,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         11,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.labelsize":    11,
    "legend.frameon":    False,
    "figure.dpi":        110,
})

PALETTE = {
    "tcn":      "#4C72B0",   # blue
    "bilstm":   "#DD8452",   # orange
    "train":    "#8C8C8C",   # grey
    "val":      "#4C72B0",   # blue
    "accent":   "#C44E52",   # red - best-epoch / mean markers
    "good":     "#55A868",   # green
}


In [2]:
VIDEO_PATH = r'data/Videos/'
LANDMARK_PATH = r'landmarks/'
PROCESSED_LANDMARK_PATH = r'landmarks_processed/'
MODEL_SAVE_PATH = r'models/saved/'
REPORT_PATH = r'VideoReport_updated.csv'
FIG_SAVE_PATH = r'plots/'

TARGET_FRAMES = 172
TARGET_FPS = 15

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(FIG_SAVE_PATH, exist_ok=True)

### Reproducibility & batching

`SEED` fixes NumPy/PyTorch RNGs and forces deterministic cuDNN kernels so re-running a cell reproduces the same split and the same training curve. `BATCH_SIZE` and `VAL_SPLIT` are kept as named constants rather than inlined so they're easy to sweep later.

In [3]:
BATCH_SIZE  = 32
VAL_SPLIT   = 0.15
NUM_WORKERS = 0 #windows 
SEED        = 42

In [4]:
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


## Skeleton Dataset & Loading

Each sample is a `(172, 59, 3)` array of landmarks (x, y, confidence). Preprocessing turns it into a `(172, 236)` tensor:

| Step | Shape | Notes |
|---|---|---|
| Drop confidence | (172, 59, 2) | Only x, y are used |
| Frame-to-frame velocity | (172, 59, 2) | First frame is zero |
| Normalize positions | (172, 59, 2) | Z-score per clip, using the mean and std over all frames and keypoints |
| Concatenate and flatten | (172, 236) | 59 keypoints x (x, y, vx, vy) = 236 |

The same `preprocess_skeleton` function is used in training and inference, so the two cannot drift apart.

In [5]:
class SkeletonDataset(Dataset):
    
    def __init__(self, root: str) -> None:
        self.samples: list[tuple[str, int]] = []
        self.classes: list[str] = sorted(
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        )
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        for cls in self.classes:
            cls_dir = os.path.join(root, cls)
            for fname in os.listdir(cls_dir):
                if fname.endswith(".npy"):
                    self.samples.append(
                        (os.path.join(cls_dir, fname), self.class_to_idx[cls])
                    )

        print(f"  Dataset: {len(self.samples)} clips, {len(self.classes)} classes")
        print(f"  Classes: {self.classes}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label = self.samples[idx]
        arr = np.load(path).astype(np.float32)#(172, 59, 3)
        arr = arr[:, :, :2] # remove conf 

        vel = np.zeros_like(arr)
        vel[1:] = arr[1:] - arr[:-1]
        arr = np.concatenate([arr, vel], axis=-1)

        #normalization
        pos = arr[:, :, :2]
        mu  = pos.mean(axis=(0, 1), keepdims=True)
        std = pos.std(axis=(0, 1), keepdims=True) + 1e-6
        arr[:, :, :2] = (pos - mu) / std

        seq = arr.reshape(172, -1)

        return torch.from_numpy(seq), torch.tensor(label, dtype=torch.long)

### Train / val / test split and class balancing

Splitting happens in two steps so the test set is held out first and never touched again: `test_idx` is carved off (stratified, so every class keeps the same proportion), then the remainder is split again into train/val. `class_weights` is computed from the *full* dataset's per-class counts and later fed into the loss function (`CrossEntropyLoss(weight=...)`) so that under-represented action classes aren't drowned out by frequent ones. Weights are clamped at 10x to stop a single very rare class from dominating the gradient.

In [6]:
full_dataset = SkeletonDataset(PROCESSED_LANDMARK_PATH)
NUM_CLASSES  = len(full_dataset.classes)

try:
    _report_n = pd.read_csv(REPORT_PATH).shape[0]
    print(f"Clips listed in {REPORT_PATH}      : {_report_n}")
except Exception as e:
    print(f"Could not read {REPORT_PATH}: {e}")
print(f"Clips actually usable (processed landmarks): {len(full_dataset)}")
if 'reports_n' not in dir() and '_report_n' in dir():
    _dropped = _report_n - len(full_dataset)
    if _dropped > 0:
        print(f"  -> {_dropped} clip(s) dropped during filtering / landmark QC "
              f"(low pose confidence, missing hands, etc.)")


# described in the paper's Training Protocol (Sec. 4.7).
TEST_SPLIT = 0.15

all_indices     = list(range(len(full_dataset)))
all_labels_list = [full_dataset.samples[i][1] for i in all_indices]

trainval_idx, test_idx = train_test_split(
    all_indices, test_size=TEST_SPLIT,
    stratify=all_labels_list, random_state=SEED,
)
trainval_labels = [all_labels_list[i] for i in trainval_idx]
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=VAL_SPLIT / (1 - TEST_SPLIT),   # keeps val at 15% of total
    stratify=trainval_labels, random_state=SEED,
)

train_ds = torch.utils.data.Subset(full_dataset, train_idx)
val_ds   = torch.utils.data.Subset(full_dataset, val_idx)
test_ds  = torch.utils.data.Subset(full_dataset, test_idx)

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, generator=g)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

print(f"\nTrain batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Number of Classes: {NUM_CLASSES}")
print(f"Train clips: {len(train_ds)} | Val clips: {len(val_ds)}")

all_labels = torch.tensor([full_dataset.samples[i][1] for i in range(len(full_dataset))])
class_counts  = torch.bincount(all_labels, minlength=NUM_CLASSES).float()
class_weights = (class_counts.sum() / (NUM_CLASSES * class_counts)).clamp(max=10.0)
print(f"Class weights: {class_weights.tolist()}")


  Dataset: 535 clips, 26 classes
  Classes: ['1-Ardhpatka', '10-Katakamukha', '11-Suchi', '12-Chandrakala', '13-Padmakosh', '14-Sarpashirsha', '15-Mrigashirsha', '16-Simhamukha', '17-Kangula', '18-Alapadma', '19-Chatura', '2-Kartarimukh', '20-Bhramara', '21-Hamsasya', '22-Hamsapkshaka', '23-Sandamsha', '24-Mukula', '25-Taamrachuda', '26-Trishula', '3-Mayura', '4-Aardhchandra', '5-Aarala', '6-Shukatundaka', '7-Mushti', '8-Shikara', '9-Kapitha']
Clips listed in VideoReport_updated.csv      : 538
Clips actually usable (processed landmarks): 535
  -> 3 clip(s) dropped during filtering / landmark QC (low pose confidence, missing hands, etc.)
Train: 373 | Val: 81 | Test: 81

Train batches: 12 | Val batches: 3
Number of Classes: 26
Train clips: 373 | Val clips: 81
Class weights: [0.6858974099159241, 1.1431623697280884, 1.1431623697280884, 1.1431623697280884, 0.709549069404602, 1.4697802066802979, 1.1431623697280884, 1.1431623697280884, 1.2104072570800781, 1.1431623697280884, 1.286057710647583

### Class balance

Before training, it's worth seeing how skewed the dataset is — a large min/max gap is exactly what `class_weights` above is meant to compensate for. The chart below sorts classes by clip count and annotates each bar with its exact count and its ratio relative to the largest class, so the imbalance is legible at a glance rather than just implied by axis height.

In [ ]:
class_names = full_dataset.classes
counts_per_class = class_counts.numpy().astype(int)

order = np.argsort(counts_per_class)[::-1]
sorted_names  = [class_names[i] for i in order]
sorted_counts = counts_per_class[order]
max_count = sorted_counts.max()

fig, ax = plt.subplots(figsize=(13, 6))

# Color bars by how imbalanced they are relative to the majority class
norm_counts = sorted_counts / max_count
colors = plt.cm.Blues(0.35 + 0.55 * norm_counts)

bars = ax.bar(range(len(sorted_names)), sorted_counts, color=colors,
              edgecolor="white", linewidth=0.6)

ax.axhline(sorted_counts.mean(), color=PALETTE["accent"], linestyle="--", linewidth=1.3,
           label=f"Mean = {sorted_counts.mean():.1f} clips/class")

# Annotate each bar with count and ratio-to-majority, only where there's room
for bar, cnt in zip(bars, sorted_counts):
    ratio = cnt / max_count
    ax.annotate(f"{cnt}\n({ratio:.0%})",
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontsize=6.5, color="#333333")

ax.set_xticks(range(len(sorted_names)))
ax.set_xticklabels(sorted_names, rotation=75, ha="right", fontsize=8)
ax.set_ylabel("Number of clips")
ax.set_title("Per-Class Clip Distribution")
ax.set_ylim(0, max_count * 1.18)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_SAVE_PATH, "class_distribution.png"), dpi=200)
plt.show()

imbalance_ratio = sorted_counts.max() / sorted_counts.min()
print(f"Min clips/class: {sorted_counts.min()}  ({sorted_names[sorted_counts.argmin()]})")
print(f"Max clips/class: {sorted_counts.max()}  ({sorted_names[sorted_counts.argmax()]})")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.1f}x")


## TCN Model

**Architecture at a glance:** a linear projection from 236 input features down to 128 channels, four residual TCN blocks with dilations `(1, 2, 4, 8)` (giving a ~60-frame receptive field), a temporal self-attention layer over the whole clip, then global average pooling into a small classification head. Causal padding is trimmed on the right after every convolution so the sequence length `T` never grows.

In [8]:
class ResidualTCNBlock(nn.Module):

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        dilation: int = 1,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        self.padding = (kernel_size - 1) * dilation

        self.conv1  = nn.Conv1d(in_channels, out_channels,
                                kernel_size=kernel_size,
                                dilation=dilation, padding=self.padding)
        self.bn1    = nn.BatchNorm1d(out_channels)
        self.relu1  = nn.ReLU(inplace=True)
        self.drop1  = nn.Dropout(p=dropout)

        self.conv2  = nn.Conv1d(out_channels, out_channels,
                                kernel_size=kernel_size,
                                dilation=dilation, padding=self.padding)
        self.bn2    = nn.BatchNorm1d(out_channels)

        self.skip = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else nn.Identity()
        )
        self.relu_out = nn.ReLU(inplace=True)
        self._init_weights()

    def _init_weights(self) -> None:
        for m in [self.conv1, self.conv2]:
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        if isinstance(self.skip, nn.Conv1d):
            nn.init.kaiming_normal_(self.skip.weight, mode="fan_out", nonlinearity="relu")

    def _causal_conv(self, conv: nn.Conv1d, x: torch.Tensor) -> torch.Tensor:
        """Conv then trim right-side causal padding to preserve T."""
        return conv(x)[:, :, :x.size(2)]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x : (B, C_in, T)
        residual = self.skip(x)                          # (B, C_out, T)

        out = self._causal_conv(self.conv1, x)           # (B, C_out, T)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.drop1(out)

        out = self._causal_conv(self.conv2, out)         # (B, C_out, T)
        out = self.bn2(out)

        out = out + residual                             # residual add
        return self.relu_out(out)                        # (B, C_out, T)


In [9]:
class TCNActionRecognizer(nn.Module):

    def __init__(
        self,
        num_classes: int,
        input_dim:   int   = 236,
        proj_dim:    int   = 128,
        kernel_size: int   = 3,
        dilations:   tuple = (1, 2, 4, 8),
        num_heads:   int   = 4,
        attn_dropout: float = 0.1,
        tcn_dropout:  float = 0.2,
        head_dropout: float = 0.4,
    ) -> None:
        super().__init__()

        # 1. Input projection  (B, T, 236) → (B, T, 128)
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(inplace=True),
        )

        # 2. Four residual TCN blocks  (B, 128, T) → (B, 128, T)
        self.tcn_blocks = nn.ModuleList([
            ResidualTCNBlock(proj_dim, proj_dim,
                             kernel_size=kernel_size,
                             dilation=d, dropout=tcn_dropout)
            for d in dilations
        ])
        # Receptive field ≈ 2 × (kernel-1) × Σ(dilations)
        #                 = 2 × 2 × 15 = 60 frames

        # 3. Temporal self-attention  (B, T, 128) → (B, T, 128)
        self.attn_norm    = nn.LayerNorm(proj_dim)
        self.temporal_attn = nn.MultiheadAttention(
            embed_dim=proj_dim, num_heads=num_heads,
            dropout=attn_dropout, batch_first=True,
        )

        # 4. Global average pool  (B, T, 128) → (B, 128)  [implicit]

        # 5. Classification head  (B, 128) → (B, num_classes)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=head_dropout),
            nn.Linear(64, num_classes),
        )

        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:  x – (B, T=172, F=236)
        Returns: logits – (B, num_classes)
        """
        # ── Input projection ──────────────────────────────────
        x = self.input_proj(x)                       # (B, 172, 128)

        # ── TCN backbone ─────────────────────────────────────
        x = x.transpose(1, 2)                        # (B, 128, 172)
        for block in self.tcn_blocks:
            x = block(x)                             # (B, 128, 172)
        x = x.transpose(1, 2)                        # (B, 172, 128)

        # ── Temporal attention ────────────────────────────────
        residual = x
        x_norm   = self.attn_norm(x)                 # (B, 172, 128)  pre-norm
        attn_out, _ = self.temporal_attn(x_norm, x_norm, x_norm)  # (B, 172, 128)
        x = attn_out + residual                      # (B, 172, 128)

        # ── Global average pool ───────────────────────────────
        x = x.mean(dim=1)                            # (B, 128)

        # ── Classification ────────────────────────────────────
        return self.classifier(x)                    # (B, num_classes)


### Training utilities

`count_parameters` / `model_summary` report model size for the write-up (used later to compare TCN vs. BiLSTM capacity). `EarlyStopping` tracks the best validation score and signals when to stop; `mode="max"` is used here because the tracked metric is validation accuracy.

In [10]:
def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def model_summary(model: nn.Module) -> None:
    total     = count_parameters(model, trainable_only=False)
    trainable = count_parameters(model, trainable_only=True)
    print("=" * 52)
    print(f"  Model          : {model.__class__.__name__}")
    print(f"  Total params   : {total:,}")
    print(f"  Trainable      : {trainable:,}")
    print(f"  Non-trainable  : {total - trainable:,}")
    print("=" * 52)


In [11]:
class EarlyStopping:
    def __init__(
        self,
        patience: int  = 10,
        min_delta: float = 1e-4,
        mode: str = "min",
        verbose: bool = True,
    ) -> None:
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.verbose    = verbose
        self.counter    = 0
        self.best_score: Optional[float] = None
        self.early_stop = False

    def __call__(self, score: float) -> bool:
        if self.best_score is None:
            self.best_score = score
            return False

        improved = (
            score < self.best_score - self.min_delta if self.mode == "min"
            else score > self.best_score + self.min_delta
        )

        if improved:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"    EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


### Epoch loop

`train_one_epoch` and `validate` implement a single pass over their respective loaders under mixed precision (`autocast` + `GradScaler`), with gradient clipping on the training side to keep the TCN's residual stack stable. `train` wires these together with AdamW, a cosine-annealed learning rate, class-weighted label-smoothed cross-entropy, and early stopping, checkpointing the best model to disk on every validation-accuracy improvement.

In [12]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: GradScaler,
    device: torch.device,
    grad_clip: float = 1.0,
) -> dict:
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    t0 = time.perf_counter()

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)   # (B, 172, 236)
        labels = labels.to(device, non_blocking=True)   # (B,)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(inputs)                      # (B, num_classes)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * inputs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    elapsed = time.perf_counter() - t0
    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
        "samples_per_sec": total / elapsed,
    }


In [13]:
@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    """Full validation pass (no gradients)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type=device.type):
            logits = model(inputs)
            loss   = criterion(logits, labels)

        total_loss += loss.item() * inputs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    return {"loss": total_loss / total, "accuracy": correct / total}


In [14]:
def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    *,
    num_classes: int,
    num_epochs: int = 100,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    grad_clip: float = 1.0,
    label_smoothing: float = 0.1,
    class_weights: Optional[torch.Tensor] = None,
    patience: int = 15,
    checkpoint_path: str = "best_tcn.pt",
    device: Optional[torch.device] = None,
) -> dict:
    """
    End-to-end training with AdamW + CosineAnnealingLR + AMP + early stopping.

    Returns history dict: train_loss, train_acc, val_loss, val_acc.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    model_summary(model)
    print(f"  Device: {device}\n")

    # Loss
    if class_weights is not None:
        class_weights = class_weights.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights,
                                    label_smoothing=label_smoothing)

    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                  weight_decay=weight_decay, betas=(0.9, 0.999))

    # Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=lr * 1e-2)

    # AMP scaler (no-op on CPU)
    scaler = GradScaler(enabled=device.type == "cuda")

    # Early stopping
    early_stop = EarlyStopping(patience=patience, mode="max", verbose=True)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0

    hdr = f"{'Epoch':>6} | {'TrLoss':>8} | {'TrAcc':>7} | {'VaLoss':>8} | {'VaAcc':>7} | {'LR':>9}"
    print(hdr)
    print("-" * len(hdr))

    for epoch in range(1, num_epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer,
                             criterion, scaler, device, grad_clip)
        va = validate(model, val_loader, criterion, device)
        scheduler.step()
        lr_now = scheduler.get_last_lr()[0]

        history["train_loss"].append(tr["loss"])
        history["train_acc"].append(tr["accuracy"])
        history["val_loss"].append(va["loss"])
        history["val_acc"].append(va["accuracy"])

        print(f"{epoch:>6} | {tr['loss']:>8.4f} | {tr['accuracy']:>6.2%} | "
              f"{va['loss']:>8.4f} | {va['accuracy']:>6.2%} | {lr_now:>9.2e}")

        # Checkpoint on improvement
        if va["accuracy"] > best_val_acc:
            best_val_acc = va["accuracy"]
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_acc": best_val_acc,
                "classes": train_loader.dataset.dataset.classes
                           if hasattr(train_loader.dataset, "dataset") else [],
            }, checkpoint_path)

        if early_stop(va["accuracy"]):
            print(f"\n  Early stopping at epoch {epoch}.")
            break

    print(f"\n  Best val accuracy: {best_val_acc:.2%}  →  '{checkpoint_path}'")
    return history


In [15]:
@torch.no_grad()
def predict(
    model: nn.Module,
    inputs: torch.Tensor,
    device: Optional[torch.device] = None,
    return_probs: bool = False
):

    if device is None:
        device = next(model.parameters()).device
    model.eval()

    if inputs.dim() == 2:
        inputs = inputs.unsqueeze(0)       # (1, T, F)

    inputs = inputs.to(device)
    with autocast(device_type=device.type, enabled=device.type == "cuda"):
        logits = model(inputs)             # (B, num_classes)

    if return_probs:
        return F.softmax(logits, dim=-1)
    return logits.argmax(dim=-1)


def load_checkpoint(
    checkpoint_path: str,
    model: nn.Module,
    device: Optional[torch.device] = None
):
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()
    print(f"Loaded checkpoint '{checkpoint_path}' "
          f"(epoch {ckpt.get('epoch', '?')}, val_acc={ckpt.get('val_acc', float('nan')):.2%})")
    return model

### Train the TCN

A quick forward pass on random data checks the output shape before committing to a full training run — a cheap way to catch a dimension mismatch before spending epochs on it.

In [16]:
torch.manual_seed(SEED)
model = TCNActionRecognizer(
    num_classes  = NUM_CLASSES,
    input_dim    = 236,
    proj_dim     = 128,
    kernel_size  = 3,
    dilations    = (1, 2, 4, 8),
    num_heads    = 4,
    attn_dropout = 0.1,
    tcn_dropout  = 0.2,
    head_dropout = 0.4,
)

# Quick shape check before training
_dummy = torch.randn(2, 172, 236)
assert model(_dummy).shape == (2, NUM_CLASSES), "Shape mismatch!"
print(f"Forward-pass shape check passed: (2, 172, 236) → (2, {NUM_CLASSES})")

history = train(
    model,
    train_loader,
    val_loader,
    num_classes     = NUM_CLASSES,
    num_epochs      = 100,
    lr              = 3e-4,
    weight_decay    = 1e-4,
    grad_clip       = 1.0,
    label_smoothing = 0.1,
    class_weights   = class_weights,   # computed in cell 4
    patience        = 15,
    checkpoint_path = os.path.join(MODEL_SAVE_PATH, "best_tcn.pt"),
    device          = DEVICE,
)

Forward-pass shape check passed: (2, 172, 236) → (2, 26)
  Model          : TCNActionRecognizer
  Total params   : 503,130
  Trainable      : 503,130
  Non-trainable  : 0
  Device: cuda

 Epoch |   TrLoss |   TrAcc |   VaLoss |   VaAcc |        LR
------------------------------------------------------------
     1 |   3.2665 |  5.36% |   3.2454 | 13.58% |  3.00e-04
     2 |   3.2326 | 11.53% |   3.1880 | 22.22% |  3.00e-04
     3 |   3.1665 | 19.03% |   3.0673 | 19.75% |  2.99e-04
    EarlyStopping counter: 1/15
     4 |   3.0777 | 20.64% |   2.9292 | 20.99% |  2.99e-04
    EarlyStopping counter: 2/15
     5 |   2.9543 | 21.45% |   2.7846 | 20.99% |  2.98e-04
    EarlyStopping counter: 3/15
     6 |   2.8255 | 24.40% |   2.6871 | 22.22% |  2.97e-04
    EarlyStopping counter: 4/15
     7 |   2.7038 | 25.47% |   2.5694 | 28.40% |  2.96e-04
     8 |   2.5740 | 27.35% |   2.4226 | 38.27% |  2.95e-04
     9 |   2.4379 | 31.90% |   2.2840 | 38.27% |  2.94e-04
    EarlyStopping counter: 1/15


### TCN training curves

Loss and accuracy are shown side by side so overfitting is easy to spot: a widening gap between train and val loss (left) alongside a plateauing or falling val accuracy (right) would flag it. The best validation-accuracy epoch is marked directly on the accuracy plot rather than left for the reader to find in a table.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

epochs = range(1, len(history["train_loss"]) + 1)
best_epoch = int(np.argmax(history["val_acc"])) + 1
best_acc   = max(history["val_acc"]) * 100

# --- Loss ---
axes[0].plot(epochs, history["train_loss"], label="Train loss", color=PALETTE["train"], linewidth=1.8)
axes[0].plot(epochs, history["val_loss"],   label="Val loss",   color=PALETTE["val"],   linewidth=1.8)
axes[0].axvline(best_epoch, color=PALETTE["accent"], linestyle=":", linewidth=1,
                label=f"Best val-acc epoch ({best_epoch})")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("TCN: Loss vs. Epoch")
axes[0].legend(fontsize=9)

# --- Accuracy ---
axes[1].plot(epochs, [a * 100 for a in history["train_acc"]], label="Train acc",
             color=PALETTE["train"], linewidth=1.8)
axes[1].plot(epochs, [a * 100 for a in history["val_acc"]],   label="Val acc",
             color=PALETTE["val"],   linewidth=1.8)
axes[1].scatter([best_epoch], [best_acc], color=PALETTE["accent"], zorder=5, s=55,
                label=f"Best val acc = {best_acc:.2f}% (epoch {best_epoch})")
axes[1].axvline(best_epoch, color=PALETTE["accent"], linestyle=":", linewidth=1)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("TCN: Accuracy vs. Epoch")
axes[1].legend(fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig(os.path.join(FIG_SAVE_PATH, "tcn_training_curves.png"), dpi=200)
plt.show()

tcn_history = history  # keep a copy for the TCN-vs-BiLSTM comparison plot later


## BiLSTM Model

**Architecture at a glance:** the same 236→128 input projection feeds a 4-layer bidirectional LSTM (hidden size 512 per direction), projected back to 128 dims, passed through one self-attention layer, then pooled and classified — mirroring the TCN's projection → sequence-mixing → attention → pool → classify shape so the two models differ mainly in *how* they mix time (dilated convolutions vs. recurrence), which is what the comparison later isolates. This BiLSTM is considerably larger than the TCN (see parameter counts below), so treat any accuracy gap alongside that capacity difference, not in isolation.

In [18]:
class BiLSTMActionRecognizer(nn.Module):
    def __init__(
        self,
        num_classes:  int,
        input_dim:    int = 236,
        proj_dim:     int = 128,
        hidden_dim:   int = 256,
        num_layers:   int = 2,
        num_heads:    int = 4,
        attn_dropout: float = 0.1,
        lstm_dropout: float = 0.3,
        head_dropout: float = 0.4,
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.ReLU(),
        )

        self.bilstm = nn.LSTM(
            input_size=proj_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=lstm_dropout if num_layers > 1 else 0.0,
        )

        self.proj = nn.Linear(hidden_dim * 2, proj_dim)

        self.attn = nn.MultiheadAttention(
            embed_dim=proj_dim,
            num_heads=num_heads,
            dropout=attn_dropout,
            batch_first=True,
        )

        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 128),
            nn.ReLU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.input_proj(x)
        x, _ = self.bilstm(x)
        x = self.proj(x)

        attn_out, _ = self.attn(x, x, x)
        x = x + attn_out

        x = x.mean(dim=1)
        return self.classifier(x)

In [19]:
def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def model_summary(model: nn.Module) -> None:
    total     = count_parameters(model, trainable_only=False)
    trainable = count_parameters(model, trainable_only=True)
    print("=" * 52)
    print(f"  Model          : {model.__class__.__name__}")
    print(f"  Total params   : {total:,}")
    print(f"  Trainable      : {trainable:,}")
    print(f"  Non-trainable  : {total - trainable:,}")
    print("=" * 52)

In [20]:
class EarlyStopping:
    def __init__(
        self,
        patience:  int   = 10,
        min_delta: float = 1e-4,
        mode:      str   = "min",
        verbose:   bool  = True,
    ) -> None:
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.verbose    = verbose
        self.counter    = 0
        self.best_score: Optional[float] = None
        self.early_stop = False

    def __call__(self, score: float) -> bool:
        if self.best_score is None:
            self.best_score = score
            return False

        improved = (
            score < self.best_score - self.min_delta if self.mode == "min"
            else score > self.best_score + self.min_delta
        )

        if improved:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"    EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

In [21]:
def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler:    GradScaler,
    device:    torch.device,
    grad_clip: float = 1.0,
) -> dict:
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    t0 = time.perf_counter()

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type):
            logits = model(inputs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * inputs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    elapsed = time.perf_counter() - t0
    return {
        "loss":            total_loss / total,
        "accuracy":        correct / total,
        "samples_per_sec": total / elapsed,
    }

In [22]:
@torch.no_grad()
def validate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
    device:    torch.device,
) -> dict:
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type=device.type):
            logits = model(inputs)
            loss   = criterion(logits, labels)

        total_loss += loss.item() * inputs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    return {
        "loss":     total_loss / total,
        "accuracy": correct / total,
    }

In [23]:
def train(
    model:           nn.Module,
    train_loader:    DataLoader,
    val_loader:      DataLoader,
    *,
    num_classes:     int,
    num_epochs:      int   = 100,
    lr:              float = 3e-4,
    weight_decay:    float = 1e-4,
    grad_clip:       float = 1.0,
    label_smoothing: float = 0.1,
    class_weights:   Optional[torch.Tensor] = None,
    patience:        int   = 15,
    checkpoint_path: str   = "best_bilstm.pt",
    device:          Optional[torch.device] = None,
) -> dict:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    model_summary(model)
    print(f"  Device: {device}\n")

    if class_weights is not None:
        class_weights = class_weights.to(device)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=label_smoothing,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.999),
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=lr * 1e-2,
    )

    scaler     = GradScaler(device="cuda" if device.type == "cuda" else "cpu")
    early_stop = EarlyStopping(patience=patience, mode="max", verbose=True)

    history    = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0

    hdr = f"{'Epoch':>6} | {'TrLoss':>8} | {'TrAcc':>7} | {'VaLoss':>8} | {'VaAcc':>7} | {'LR':>9}"
    print(hdr)
    print("-" * len(hdr))

    for epoch in range(1, num_epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer,
                             criterion, scaler, device, grad_clip)
        va = validate(model, val_loader, criterion, device)
        scheduler.step()
        lr_now = scheduler.get_last_lr()[0]

        history["train_loss"].append(tr["loss"])
        history["train_acc"].append(tr["accuracy"])
        history["val_loss"].append(va["loss"])
        history["val_acc"].append(va["accuracy"])

        print(f"{epoch:>6} | {tr['loss']:>8.4f} | {tr['accuracy']:>6.2%} | "
              f"{va['loss']:>8.4f} | {va['accuracy']:>6.2%} | {lr_now:>9.2e}")

        if va["accuracy"] > best_val_acc:
            best_val_acc = va["accuracy"]
            torch.save({
                "epoch":              epoch,
                "model_state_dict":   model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_acc":            best_val_acc,
                "classes":            train_loader.dataset.dataset.classes
                                      if hasattr(train_loader.dataset, "dataset") else [],
            }, checkpoint_path)

        if early_stop(va["accuracy"]):
            print(f"\n  Early stopping at epoch {epoch}.")
            break

    print(f"\n  Best val accuracy: {best_val_acc:.2%}  →  '{checkpoint_path}'")
    return history

In [24]:
@torch.no_grad()
def predict_from_path_bilstm(
    npy_path:        str,
    checkpoint_path: str,
    classes:         list[str],
    num_classes:     Optional[int] = None,
    device:          Optional[torch.device] = None,
) -> dict:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if num_classes is None:
        num_classes = len(classes)

    ckpt  = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state = ckpt["model_state_dict"]

    # Infer architecture from the checkpoint so it always matches training
    proj_dim, input_dim = state["input_proj.0.weight"].shape
    hidden_dim = state["bilstm.weight_hh_l0"].shape[1]
    num_layers = sum(
        1 for k in state
        if k.startswith("bilstm.weight_ih_l") and not k.endswith("_reverse")
    )
    ckpt_classes = state["classifier.3.weight"].shape[0]
    assert ckpt_classes == num_classes, (
        f"Checkpoint has {ckpt_classes} classes, got {num_classes}"
    )

    model = BiLSTMActionRecognizer(
        num_classes=num_classes,
        input_dim=input_dim,
        proj_dim=proj_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        num_heads=4,          # not recoverable from weights; must match training
        attn_dropout=0.1,
        lstm_dropout=0.2,     # inactive in eval, does not affect loading
        head_dropout=0.3,     # inactive in eval, does not affect loading
    )
    model.load_state_dict(state)
    model.to(device).eval()

    # Same preprocessing as SkeletonDataset.__getitem__
    arr = np.load(npy_path).astype(np.float32)          # (172, 59, 3)
    arr = arr[:, :, :2]                                 # (172, 59, 2)
    vel = np.zeros_like(arr)
    vel[1:] = arr[1:] - arr[:-1]
    arr = np.concatenate([arr, vel], axis=-1)           # (172, 59, 4)
    pos = arr[:, :, :2]
    mu  = pos.mean(axis=(0, 1), keepdims=True)
    std = pos.std(axis=(0, 1), keepdims=True) + 1e-6
    arr[:, :, :2] = (pos - mu) / std
    seq    = arr.reshape(172, -1)                       # (172, 236)
    tensor = torch.from_numpy(seq).unsqueeze(0).to(device)

    with autocast(device_type=device.type, enabled=device.type == "cuda"):
        logits = model(tensor)

    probs = F.softmax(logits.float(), dim=-1).squeeze(0)
    top3_vals, top3_idxs = probs.topk(min(3, num_classes))
    top3 = [
        {"class": classes[i.item()], "idx": i.item(), "confidence": round(v.item(), 4)}
        for v, i in zip(top3_vals, top3_idxs)
    ]

    return {
        "predicted_class": top3[0]["class"],
        "predicted_idx":   top3[0]["idx"],
        "confidence":      top3[0]["confidence"],
        "top3":            top3,
        "all_probs":       {classes[i]: round(probs[i].item(), 4) for i in range(num_classes)},
    }

### Train the BiLSTM

Same shape check as before, then training with its own learning-rate/grad-clip settings (`lr=1e-3`, `grad_clip=0.5`) — tuned separately from the TCN since the two architectures respond differently to step size and gradient scale.

In [25]:

torch.manual_seed(SEED)
model_bilstm = BiLSTMActionRecognizer(
    num_classes  = NUM_CLASSES,
    input_dim    = 236,
    proj_dim     = 128,
    hidden_dim   = 512,
    num_layers   = 4,
    num_heads    = 4,
    attn_dropout = 0.1,
    lstm_dropout = 0.2,
    head_dropout = 0.3,
)

_dummy = torch.randn(2, 172, 236)
assert model_bilstm(_dummy).shape == (2, NUM_CLASSES), "Shape mismatch!"
print(f"Forward-pass check passed: (2, 172, 236) → (2, {NUM_CLASSES})")

history_bilstm = train(
    model_bilstm,
    train_loader,
    val_loader,
    num_classes     = NUM_CLASSES,
    num_epochs      = 100,
    lr              = 1e-3,
    weight_decay    = 1e-4,
    grad_clip       = 0.5,
    label_smoothing = 0.1,
    class_weights   = class_weights,
    patience        = 15,
    checkpoint_path = "models/saved/best_bilstm.pt",
    device          = DEVICE,
)

Forward-pass check passed: (2, 172, 236) → (2, 26)
  Model          : BiLSTMActionRecognizer
  Total params   : 21,776,026
  Trainable      : 21,776,026
  Non-trainable  : 0
  Device: cuda

 Epoch |   TrLoss |   TrAcc |   VaLoss |   VaAcc |        LR
------------------------------------------------------------
     1 |   3.3135 |  4.29% |   3.2708 |  3.70% |  1.00e-03
     2 |   3.2710 |  5.36% |   3.2248 |  6.17% |  9.99e-04
     3 |   3.1893 |  8.31% |   3.0665 |  4.94% |  9.98e-04
    EarlyStopping counter: 1/15
     4 |   3.0564 |  7.24% |   2.9374 | 11.11% |  9.96e-04
     5 |   2.9842 |  9.92% |   2.8526 | 11.11% |  9.94e-04
    EarlyStopping counter: 1/15
     6 |   2.8878 | 11.80% |   2.8287 | 12.35% |  9.91e-04
     7 |   2.8858 | 12.60% |   2.7589 | 14.81% |  9.88e-04
     8 |   2.7633 | 14.21% |   2.7327 | 14.81% |  9.84e-04
    EarlyStopping counter: 1/15
     9 |   2.7594 | 15.82% |   2.6162 | 22.22% |  9.80e-04
    10 |   2.6133 | 17.16% |   2.5147 | 24.69% |  9.76e-04
  

### BiLSTM training curves

Same layout as the TCN curves above, so the two figures can be compared panel-for-panel before the direct overlay in the next section.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

epochs_b = range(1, len(history_bilstm["train_loss"]) + 1)
best_epoch_b = int(np.argmax(history_bilstm["val_acc"])) + 1
best_acc_b   = max(history_bilstm["val_acc"]) * 100

# --- Loss ---
axes[0].plot(epochs_b, history_bilstm["train_loss"], label="Train loss", color=PALETTE["train"], linewidth=1.8)
axes[0].plot(epochs_b, history_bilstm["val_loss"],   label="Val loss",   color=PALETTE["bilstm"], linewidth=1.8)
axes[0].axvline(best_epoch_b, color=PALETTE["accent"], linestyle=":", linewidth=1,
                label=f"Best val-acc epoch ({best_epoch_b})")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("BiLSTM: Loss vs. Epoch")
axes[0].legend(fontsize=9)

# --- Accuracy ---
axes[1].plot(epochs_b, [a * 100 for a in history_bilstm["train_acc"]], label="Train acc",
             color=PALETTE["train"], linewidth=1.8)
axes[1].plot(epochs_b, [a * 100 for a in history_bilstm["val_acc"]],   label="Val acc",
             color=PALETTE["bilstm"], linewidth=1.8)
axes[1].scatter([best_epoch_b], [best_acc_b], color=PALETTE["accent"], zorder=5, s=55,
                label=f"Best val acc = {best_acc_b:.2f}% (epoch {best_epoch_b})")
axes[1].axvline(best_epoch_b, color=PALETTE["accent"], linestyle=":", linewidth=1)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("BiLSTM: Accuracy vs. Epoch")
axes[1].legend(fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig(os.path.join(FIG_SAVE_PATH, "bilstm_training_curves.png"), dpi=200)
plt.show()

bilstm_history = history_bilstm  # keep a copy for the comparison plot below


## Evaluation

### TCN vs. BiLSTM

The two models are compared on three axes at once: validation-accuracy trajectory over training (left), final loss trajectory (middle, on a log scale since loss typically spans an order of magnitude early vs. late in training), and a parameter-count-vs-best-accuracy summary (right) so a large model's win can be weighed against how many more parameters it needed to get there.

In [ ]:
TCN_PARAMS    = 503_130
BILSTM_PARAMS = 21_776_026

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# --- Panel 1: validation accuracy ---
axes[0].plot(range(1, len(tcn_history["val_acc"]) + 1),
             [a * 100 for a in tcn_history["val_acc"]],
             label=f"TCN - best {max(tcn_history['val_acc'])*100:.2f}%",
             color=PALETTE["tcn"], linewidth=2)
axes[0].plot(range(1, len(bilstm_history["val_acc"]) + 1),
             [a * 100 for a in bilstm_history["val_acc"]],
             label=f"BiLSTM - best {max(bilstm_history['val_acc'])*100:.2f}%",
             color=PALETTE["bilstm"], linewidth=2)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Validation accuracy (%)")
axes[0].set_title("Validation Accuracy")
axes[0].legend(fontsize=9)

# --- Panel 2: validation loss (log scale) ---
axes[1].plot(range(1, len(tcn_history["val_loss"]) + 1),
             tcn_history["val_loss"], label="TCN", color=PALETTE["tcn"], linewidth=2)
axes[1].plot(range(1, len(bilstm_history["val_loss"]) + 1),
             bilstm_history["val_loss"], label="BiLSTM", color=PALETTE["bilstm"], linewidth=2)
axes[1].set_yscale("log")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation loss (log scale)")
axes[1].set_title("Validation Loss")
axes[1].legend(fontsize=9)

# --- Panel 3: parameter count vs. best accuracy ---
models_lbl = ["TCN", "BiLSTM"]
params     = [TCN_PARAMS, BILSTM_PARAMS]
best_accs  = [max(tcn_history["val_acc"]) * 100, max(bilstm_history["val_acc"]) * 100]
bar_colors = [PALETTE["tcn"], PALETTE["bilstm"]]

ax3 = axes[2]
bars = ax3.bar(models_lbl, best_accs, color=bar_colors, width=0.55)
for bar, p, acc in zip(bars, params, best_accs):
    ax3.annotate(f"{acc:.2f}%\n{p/1e6:.1f}M params",
                 xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                 xytext=(0, 4), textcoords="offset points",
                 ha="center", va="bottom", fontsize=9)
ax3.set_ylabel("Best validation accuracy (%)")
ax3.set_title("Accuracy vs. Model Size")
ax3.set_ylim(0, max(best_accs) * 1.25)

plt.tight_layout()
plt.savefig(os.path.join(FIG_SAVE_PATH, "tcn_vs_bilstm.png"), dpi=200)
plt.show()

print(f"TCN params    : {TCN_PARAMS:,}")
print(f"BiLSTM params : {BILSTM_PARAMS:,}  ({BILSTM_PARAMS/TCN_PARAMS:.1f}x larger)")
print(f"TCN best val acc    : {max(tcn_history['val_acc'])*100:.2f}%")
print(f"BiLSTM best val acc : {max(bilstm_history['val_acc'])*100:.2f}%")


### Per-class evaluation

`evaluate` runs a full pass with no gradient tracking and returns overall accuracy, average loss, a per-class precision/recall/F1 breakdown, and the raw confusion matrix — everything needed for the plots and tables below, computed once rather than re-derived per plot.

In [28]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    classes: list[str],
    device: Optional[torch.device] = None,
) -> dict:
    """
    Full evaluation pass with per-class metrics.

    Args:
        model   : Trained TCNActionRecognizer.
        loader  : DataLoader for the evaluation set.
        classes : Ordered list of class names.
        device  : Target device. Auto-detected if None.

    Returns:
        dict with keys:
            accuracy        (float)       – overall accuracy
            avg_loss        (float)       – average cross-entropy loss
            class_report    (dict)        – per-class precision, recall, f1, support
            confusion_matrix (np.ndarray) – (num_classes, num_classes)
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    num_classes = len(classes)
    criterion   = nn.CrossEntropyLoss()

    model.to(device).eval()

    all_preds  = []
    all_labels = []
    total_loss = 0.0
    total      = 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(inputs)
            loss   = criterion(logits, labels)

        total_loss += loss.item() * inputs.size(0)
        total      += inputs.size(0)

        all_preds.append(logits.argmax(dim=1).cpu())
        all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    # ── Overall accuracy ───────────────────────────────────────────────────
    accuracy = (all_preds == all_labels).float().mean().item()

    # ── Confusion matrix ───────────────────────────────────────────────────
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    for true, pred in zip(all_labels.tolist(), all_preds.tolist()):
        confusion[true][pred] += 1

    # ── Per-class precision, recall, F1 ───────────────────────────────────
    class_report = {}
    for i, cls in enumerate(classes):
        tp = confusion[i, i]
        fp = confusion[:, i].sum() - tp
        fn = confusion[i, :].sum() - tp

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)

        class_report[cls] = {
            "precision": round(precision, 4),
            "recall":    round(recall,    4),
            "f1":        round(f1,        4),
            "support":   int(confusion[i, :].sum()),
        }

    # ── Print summary ──────────────────────────────────────────────────────
    print(f"\n{'='*58}")
    print(f"  Overall accuracy : {accuracy:.2%}")
    print(f"  Average loss     : {total_loss / total:.4f}")
    print(f"{'='*58}")
    print(f"  {'Class':<20} {'Prec':>6} {'Rec':>6} {'F1':>6} {'N':>5}")
    print(f"  {'-'*47}")
    for cls, m in class_report.items():
        print(f"  {cls:<20} {m['precision']:>6.2%} {m['recall']:>6.2%} "
              f"{m['f1']:>6.2%} {m['support']:>5}")
    print(f"{'='*58}\n")

    return {
        "accuracy":         accuracy,
        "avg_loss":         round(total_loss / total, 4),
        "class_report":     class_report,
        "confusion_matrix": confusion,
    }

In [29]:
eval_model = TCNActionRecognizer(
    num_classes  = NUM_CLASSES,
    input_dim    = 236,
    proj_dim     = 128,
    kernel_size  = 3,
    dilations    = (1, 2, 4, 8),
    num_heads    = 4,
    attn_dropout = 0.1,
    tcn_dropout  = 0.2,
    head_dropout = 0.4,
)
eval_model = load_checkpoint(os.path.join(MODEL_SAVE_PATH, "best_tcn.pt"),
                              eval_model, device=DEVICE)

results = evaluate(eval_model, val_loader, classes=full_dataset.classes, device=DEVICE)

cm_df = pd.DataFrame(
    results["confusion_matrix"],
    index=full_dataset.classes,
    columns=full_dataset.classes,
)
cm_df.to_csv(os.path.join(FIG_SAVE_PATH, "confusion_matrix.csv"))
cm_df


Loaded checkpoint 'models/saved/best_tcn.pt' (epoch 57, val_acc=95.06%)

  Overall accuracy : 95.06%
  Average loss     : 0.2747
  Class                  Prec    Rec     F1     N
  -----------------------------------------------
  1-Ardhpatka          100.00% 80.00% 88.89%     5
  10-Katakamukha       100.00% 100.00% 100.00%     3
  11-Suchi             100.00% 100.00% 100.00%     3
  12-Chandrakala       100.00% 100.00% 100.00%     3
  13-Padmakosh         100.00% 50.00% 66.67%     4
  14-Sarpashirsha      50.00% 100.00% 66.67%     2
  15-Mrigashirsha      100.00% 66.67% 80.00%     3
  16-Simhamukha        75.00% 100.00% 85.71%     3
  17-Kangula           100.00% 100.00% 100.00%     3
  18-Alapadma          100.00% 100.00% 100.00%     3
  19-Chatura           100.00% 100.00% 100.00%     2
  2-Kartarimukh        100.00% 100.00% 100.00%     3
  20-Bhramara          100.00% 100.00% 100.00%     4
  21-Hamsasya          100.00% 100.00% 100.00%     3
  22-Hamsapkshaka      100.00% 100.00% 

,1-Ardhpatka,10-Katakamukha,11-Suchi,12-Chandrakala,13-Padmakosh,14-Sarpashirsha,15-Mrigashirsha,16-Simhamukha,17-Kangula,18-Alapadma,...,24-Mukula,25-Taamrachuda,26-Trishula,3-Mayura,4-Aardhchandra,5-Aarala,6-Shukatundaka,7-Mushti,8-Shikara,9-Kapitha
1-Ardhpatka,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
10-Katakamukha,0,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
11-Suchi,0,0,3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12-Chandrakala,0,0,0,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13-Padmakosh,0,0,0,0,2,2,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
14-Sarpashirsha,0,0,0,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
15-Mrigashirsha,0,0,0,0,0,0,2,1,0,0,...,0,0,0,0,0,0,0,0,0,0
16-Simhamukha,0,0,0,0,0,0,0,3,0,0,...,0,0,0,0,0,0,0,0,0,0
17-Kangula,0,0,0,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
18-Alapadma,0,0,0,0,0,0,0,0,0,3,...,0,0,0,0,0,0,0,0,0,0


### Per-class F1

Sorted ascending so the weakest classes — the ones most worth discussing in error analysis — sit at the top. Precision and recall are added alongside F1 as thin markers, so a low F1 caused by low precision (over-predicting that class) reads differently from one caused by low recall (missing that class), which the bar alone couldn't distinguish.

In [ ]:
report_df = pd.DataFrame(results["class_report"]).T
report_df = report_df[["precision", "recall", "f1", "support"]].sort_values("f1")
report_df.to_csv(os.path.join(FIG_SAVE_PATH, "per_class_metrics.csv"))

print(f"Macro-avg precision: {report_df['precision'].mean():.4f}")
print(f"Macro-avg recall   : {report_df['recall'].mean():.4f}")
print(f"Macro-avg F1        : {report_df['f1'].mean():.4f}")
display(report_df)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
y = np.arange(len(report_df))

ax.barh(y, report_df["f1"], color=PALETTE["good"], alpha=0.85, label="F1", zorder=2)
ax.scatter(report_df["precision"], y, color="#333333", marker="|", s=90,
           linewidths=1.6, label="Precision", zorder=3)
ax.scatter(report_df["recall"], y, color=PALETTE["accent"], marker="|", s=90,
           linewidths=1.6, label="Recall", zorder=3)

ax.set_yticks(y); ax.set_yticklabels(report_df.index, fontsize=8)
ax.set_xlabel("Score")
ax.set_xlim(0, 1.05)
ax.set_title("Per-Class F1 Score (TCN), with Precision/Recall markers")
ax.axvline(report_df["f1"].mean(), color=PALETTE["accent"], linestyle="--",
           label=f"Macro-avg F1 = {report_df['f1'].mean():.2f}")
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(FIG_SAVE_PATH, "fig_per_class_f1.png"), dpi=200)
plt.show()


### Confusion matrix

Raw counts are hard to compare across classes with very different support, so the matrix below is shown **row-normalized** (each row sums to 1, i.e. "of the true clips of this class, what fraction went to each predicted class") alongside the original per-cell counts as text annotations. The most-confused off-diagonal pairs are printed underneath for the discussion section.

In [ ]:
cm = results["confusion_matrix"]
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

# Annotate each cell with the raw count (skip zeros to reduce clutter)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j] > 0:
            text_color = "white" if cm_norm[i, j] > 0.5 else "#333333"
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    fontsize=6, color=text_color)

ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(full_dataset.classes, rotation=90, fontsize=7)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(full_dataset.classes, fontsize=7)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title(f"Confusion Matrix - TCN, row-normalized (val accuracy {results['accuracy']*100:.2f}%)")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Fraction of true class")
plt.tight_layout()
plt.savefig(os.path.join(FIG_SAVE_PATH, "fig_confusion_matrix.png"), dpi=200)
plt.show()

# Top confused class pairs (off-diagonal), useful for the Discussion section
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm_off[i, j] > 0:
            pairs.append((full_dataset.classes[i], full_dataset.classes[j], int(cm_off[i, j])))
pairs.sort(key=lambda x: -x[2])
print("Top confused (true -> predicted) pairs:")
for true_c, pred_c, n in pairs[:10]:
    print(f"  {true_c:<20} -> {pred_c:<20} : {n} clip(s)")
